# 23. 动量与均值回归策略

## 学习目标

通过本次学习，你将能够：

1. **理解动量策略的金融逻辑**
2. **实现 Jegadeesh-Titman 动量策略**
3. **实现布林带均值回归策略**
4. **实现 RSI 策略**
5. **理解动量崩溃风险**
6. **添加仓位管理和止损机制**

## 知识地图

```
动量与均值回归
├── 动量策略
│   ├── 金融逻辑：强者恒强
│   ├── Jegadeesh-Titman 动量
│   ├── 动量崩溃风险
│   └── 仓位管理与止损
├── 均值回归策略
│   ├── 金融逻辑：价格终将回归
│   ├── 布林带策略
│   ├── RSI 策略
│   └── 仓位管理与止损
└── 策略对比
    ├── 夏普比
    ├── 最大回撤
    └── 适用市场环境
```

## 环境依赖

```bash
pip install numpy pandas matplotlib
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline

---
## 1. 理论基础

### 1.1 动量策略

**动量效应**（Momentum Effect）是指：**过去表现好的股票，未来表现也好；过去表现差的股票，未来表现也差**。

#### 金融逻辑

1. **行为金融学解释**：
   - 投资者对信息反应不足（Underreaction）
   - 信息逐渐被市场消化，价格缓慢调整
   - 赢家继续涨，输家继续跌

2. **正反馈循环**：
   - 价格上涨 → 更多投资者关注 → 更多买入 → 价格继续上涨
   - 形成自我强化的趋势

#### Jegadeesh-Titman 动量

这是最经典的动量策略（1993年提出）：

1. **形成期**（Formation Period）：过去 3-12 个月的收益率
2. **持有期**（Holding Period）：未来 3-12 个月
3. **策略**：买入过去赢家，卖空过去输家

### 1.2 均值回归策略

**均值回归**（Mean Reversion）是指：**价格偏离均值后，最终会回归均值**。

#### 金融逻辑

1. **套利机制**：
   - 价格过高 → 卖出压力 → 价格下跌
   - 价格过低 → 买入压力 → 价格上涨

2. **基本面锚定**：
   - 股票价格最终由基本面决定
   - 偏离基本面的价格不可持续

### 1.3 动量 vs 均值回归

| 策略 | 逻辑 | 适用环境 | 风险 |
|------|------|----------|------|
| **动量** | 强者恒强 | 趋势市场 | 动量崩溃 |
| **均值回归** | 价格回归 | 震荡市场 | 趋势延续 |

**关键洞察**：两种策略在不同市场环境下表现不同，可以互补。

---
## 2. 数据准备

In [ ]:
def generate_stock_data(n_stocks=5, n_periods=500):
    """
    生成模拟股票数据
    """
    np.random.seed(42)
    
    dates = pd.date_range('2020-01-01', periods=n_periods, freq='B')
    stock_names = ['股票A', '股票B', '股票C', '股票D', '股票E']
    
    # 生成带有动量和均值回归特征的价格
    prices = {}
    for i, stock in enumerate(stock_names):
        # 基础价格过程（带趋势和波动）
        returns = np.random.normal(0.0003, 0.02, n_periods)
        
        # 添加动量效应（自相关）
        momentum = 0.3
        for t in range(1, n_periods):
            returns[t] += momentum * returns[t-1]
        
        # 添加均值回归效应
        mean_reversion = -0.05
        cum_returns = np.cumsum(returns)
        returns += mean_reversion * cum_returns / n_periods
        
        # 生成价格
        price = 100 * np.exp(np.cumsum(returns))
        prices[stock] = price
    
    prices_df = pd.DataFrame(prices, index=dates)
    returns_df = prices_df.pct_change().dropna()
    
    return prices_df, returns_df


# 生成数据
prices_df, returns_df = generate_stock_data()

print(f"股票数据：")
print(f"  时间范围: {prices_df.index[0]} 到 {prices_df.index[-1]}")
print(f"  股票数量: {len(prices_df.columns)}")
print(f"  数据点数: {len(prices_df)}")

# 绘制价格走势
fig, ax = plt.subplots(figsize=(12, 6))
for stock in prices_df.columns:
    ax.plot(prices_df.index, prices_df[stock], label=stock, linewidth=1.5)
ax.set_xlabel('日期')
ax.set_ylabel('价格')
ax.set_title('股票价格走势')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. 动量策略实现

### 3.1 Jegadeesh-Titman 动量策略

**策略逻辑**：
1. 计算过去 N 天的收益率（形成期）
2. 按收益率排序，买入赢家，卖空输家
3. 持有 M 天后调仓（持有期）

In [ ]:
def momentum_strategy(prices_df, lookback=20, holding=5, top_n=2):
    """
    动量策略
    
    Parameters
    ----------
    prices_df : DataFrame - 价格数据
    lookback : int - 形成期（回看天数）
    holding : int - 持有期
    top_n : int - 买入的股票数量
    
    Returns
    -------
    DataFrame : 策略收益
    """
    returns_df = prices_df.pct_change()
    n_stocks = len(prices_df.columns)
    
    # 初始化
    positions = pd.DataFrame(0, index=prices_df.index, columns=prices_df.columns)
    strategy_returns = pd.Series(0, index=prices_df.index, dtype=float)
    
    # 交易日计数
    trade_day = 0
    
    for i in range(lookback, len(prices_df)):
        # 每 holding 天调仓一次
        if trade_day % holding == 0:
            # 计算过去 lookback 天的收益率
            past_returns = returns_df.iloc[i-lookback:i].sum()
            
            # 排序，选择赢家
            ranked = past_returns.sort_values(ascending=False)
            winners = ranked.head(top_n).index.tolist()
            losers = ranked.tail(top_n).index.tolist()
            
            # 设置仓位：买入赢家，卖空输家
            positions.iloc[i] = 0
            positions.iloc[i][winners] = 1 / top_n  # 做多赢家
            positions.iloc[i][losers] = -1 / top_n   # 做空输家
        else:
            # 保持仓位
            positions.iloc[i] = positions.iloc[i-1]
        
        # 计算收益
        strategy_returns.iloc[i] = (positions.iloc[i-1] * returns_df.iloc[i]).sum()
        trade_day += 1
    
    return strategy_returns, positions


# 运行动量策略
momentum_returns, momentum_positions = momentum_strategy(prices_df, lookback=20, holding=5)

# 计算累积收益
momentum_cumulative = (1 + momentum_returns).cumprod()

print(f"动量策略参数：")
print(f"  形成期: 20 天")
print(f"  持有期: 5 天")
print(f"  买入数量: 2 只")
print(f"\n策略表现：")
print(f"  总收益: {momentum_cumulative.iloc[-1] - 1:.2%}")
print(f"  年化收益: {(momentum_cumulative.iloc[-1] ** (252/len(prices_df))) - 1:.2%}")
print(f"  年化波动: {momentum_returns.std() * np.sqrt(252):.2%}")
print(f"  夏普比: {(momentum_returns.mean() * 252) / (momentum_returns.std() * np.sqrt(252)):.2f}")

### 3.2 动量崩溃风险

**动量崩溃**（Momentum Crash）是指动量策略在市场反转时遭受巨大损失。

#### 为什么会发生动量崩溃？

1. **市场反转**：牛市转熊市时，过去赢家变成输家
2. **拥挤交易**：太多人使用动量策略，导致踩踏
3. **流动性危机**：市场恐慌时，赢家被抛售

#### 如何应对？

1. **止损**：设定最大亏损阈值
2. **波动率调整**：高波动时降低仓位
3. **市场状态识别**：识别市场趋势，避免在反转时使用动量

In [ ]:
def momentum_strategy_with_stoploss(prices_df, lookback=20, holding=5, top_n=2, stop_loss=0.05):
    """
    带止损的动量策略
    
    Parameters
    ----------
    stop_loss : float - 止损阈值（如 0.05 表示 5%）
    """
    returns_df = prices_df.pct_change()
    n_stocks = len(prices_df.columns)
    
    positions = pd.DataFrame(0, index=prices_df.index, columns=prices_df.columns)
    strategy_returns = pd.Series(0, index=prices_df.index, dtype=float)
    
    trade_day = 0
    cumulative_loss = 0
    is_stopped = False
    
    for i in range(lookback, len(prices_df)):
        # 检查止损
        if cumulative_loss < -stop_loss:
            is_stopped = True
            positions.iloc[i] = 0
            strategy_returns.iloc[i] = 0
            
            # 止损后等待 10 天恢复
            if cumulative_loss < -stop_loss * 2:
                cumulative_loss = 0
                is_stopped = False
            continue
        
        # 正常交易
        if trade_day % holding == 0:
            past_returns = returns_df.iloc[i-lookback:i].sum()
            ranked = past_returns.sort_values(ascending=False)
            winners = ranked.head(top_n).index.tolist()
            losers = ranked.tail(top_n).index.tolist()
            
            positions.iloc[i] = 0
            positions.iloc[i][winners] = 1 / top_n
            positions.iloc[i][losers] = -1 / top_n
        else:
            positions.iloc[i] = positions.iloc[i-1]
        
        strategy_returns.iloc[i] = (positions.iloc[i-1] * returns_df.iloc[i]).sum()
        cumulative_loss = (1 + cumulative_loss) * (1 + strategy_returns.iloc[i]) - 1
        trade_day += 1
    
    return strategy_returns, positions


# 运行带止损的动量策略
momentum_sl_returns, momentum_sl_positions = momentum_strategy_with_stoploss(
    prices_df, lookback=20, holding=5, stop_loss=0.05
)
momentum_sl_cumulative = (1 + momentum_sl_returns).cumprod()

print(f"带止损的动量策略：")
print(f"  止损阈值: 5%")
print(f"  总收益: {momentum_sl_cumulative.iloc[-1] - 1:.2%}")
print(f"  夏普比: {(momentum_sl_returns.mean() * 252) / (momentum_sl_returns.std() * np.sqrt(252)):.2f}")

---
## 4. 布林带均值回归策略

### 4.1 布林带原理

**布林带**（Bollinger Bands）由三条线组成：

- **中轨**：N 日移动平均线
- **上轨**：中轨 + K × N 日标准差
- **下轨**：中轨 - K × N 日标准差

**策略逻辑**：
- 价格触及下轨 → 超卖 → 买入
- 价格触及上轨 → 超买 → 卖出

In [ ]:
def bollinger_strategy(prices_df, window=20, num_std=2, stock='股票A'):
    """
    布林带均值回归策略
    
    Parameters
    ----------
    window : int - 移动平均窗口
    num_std : int - 标准差倍数
    stock : str - 交易的股票
    """
    prices = prices_df[stock]
    returns = prices.pct_change()
    
    # 计算布林带
    ma = prices.rolling(window=window).mean()
    std = prices.rolling(window=window).std()
    upper = ma + num_std * std
    lower = ma - num_std * std
    
    # 初始化
    positions = pd.Series(0, index=prices.index, dtype=float)
    strategy_returns = pd.Series(0, index=prices.index, dtype=float)
    
    # 交易逻辑
    for i in range(window, len(prices)):
        if prices.iloc[i] < lower.iloc[i]:
            # 价格触及下轨，买入
            positions.iloc[i] = 1
        elif prices.iloc[i] > upper.iloc[i]:
            # 价格触及上轨，卖出
            positions.iloc[i] = -1
        else:
            # 保持仓位
            positions.iloc[i] = positions.iloc[i-1]
        
        # 计算收益
        strategy_returns.iloc[i] = positions.iloc[i-1] * returns.iloc[i]
    
    return strategy_returns, positions, ma, upper, lower


# 运行布林带策略
bb_returns, bb_positions, ma, upper, lower = bollinger_strategy(prices_df, window=20, num_std=2)
bb_cumulative = (1 + bb_returns).cumprod()

print(f"布林带策略参数：")
print(f"  窗口: 20 天")
print(f"  标准差倍数: 2")
print(f"\n策略表现：")
print(f"  总收益: {bb_cumulative.iloc[-1] - 1:.2%}")
print(f"  年化收益: {(bb_cumulative.iloc[-1] ** (252/len(prices_df))) - 1:.2%}")
print(f"  夏普比: {(bb_returns.mean() * 252) / (bb_returns.std() * np.sqrt(252)):.2f}")

# 绘制布林带
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# 价格和布林带
ax1 = axes[0]
ax1.plot(prices_df.index, prices_df['股票A'], label='价格', linewidth=1.5)
ax1.plot(prices_df.index, ma, label='中轨', linestyle='--', alpha=0.7)
ax1.fill_between(prices_df.index, upper, lower, alpha=0.2, color='gray', label='布林带')
ax1.set_xlabel('日期')
ax1.set_ylabel('价格')
ax1.set_title('布林带策略')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 仓位
ax2 = axes[1]
ax2.plot(prices_df.index, bb_positions, label='仓位', linewidth=1)
ax2.set_xlabel('日期')
ax2.set_ylabel('仓位')
ax2.set_title('布林带策略仓位')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. RSI 策略

### 5.1 RSI 原理

**RSI（Relative Strength Index）** 是衡量股票超买超卖的指标：

$$RSI = 100 - \frac{100}{1 + RS}$$

其中 $RS = \frac{平均上涨幅度}{平均下跌幅度}$

**策略逻辑**：
- RSI < 30 → 超卖 → 买入
- RSI > 70 → 超买 → 卖出

In [ ]:
def rsi_strategy(prices_df, window=14, overbought=70, oversold=30, stock='股票A'):
    """
    RSI 策略
    
    Parameters
    ----------
    window : int - RSI 计算窗口
    overbought : int - 超买阈值
    oversold : int - 超卖阈值
    """
    prices = prices_df[stock]
    returns = prices.pct_change()
    
    # 计算 RSI
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    
    # 初始化
    positions = pd.Series(0, index=prices.index, dtype=float)
    strategy_returns = pd.Series(0, index=prices.index, dtype=float)
    
    # 交易逻辑
    for i in range(window, len(prices)):
        if rsi.iloc[i] < oversold:
            # 超卖，买入
            positions.iloc[i] = 1
        elif rsi.iloc[i] > overbought:
            # 超买，卖出
            positions.iloc[i] = -1
        else:
            # 保持仓位
            positions.iloc[i] = positions.iloc[i-1]
        
        strategy_returns.iloc[i] = positions.iloc[i-1] * returns.iloc[i]
    
    return strategy_returns, positions, rsi


# 运行 RSI 策略
rsi_returns, rsi_positions, rsi = rsi_strategy(prices_df, window=14)
rsi_cumulative = (1 + rsi_returns).cumprod()

print(f"RSI 策略参数：")
print(f"  窗口: 14 天")
print(f"  超买阈值: 70")
print(f"  超卖阈值: 30")
print(f"\n策略表现：")
print(f"  总收益: {rsi_cumulative.iloc[-1] - 1:.2%}")
print(f"  年化收益: {(rsi_cumulative.iloc[-1] ** (252/len(prices_df))) - 1:.2%}")
print(f"  夏普比: {(rsi_returns.mean() * 252) / (rsi_returns.std() * np.sqrt(252)):.2f}")

---
## 6. 策略对比

### 6.1 绩效指标对比

In [ ]:
def calculate_metrics(returns, name):
    """
    计算策略绩效指标
    """
    cumulative = (1 + returns).cumprod()
    total_return = cumulative.iloc[-1] - 1
    annual_return = (cumulative.iloc[-1] ** (252/len(returns))) - 1
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = (returns.mean() * 252) / annual_vol if annual_vol > 0 else 0
    
    # 最大回撤
    peak = cumulative.expanding().max()
    drawdown = (cumulative - peak) / peak
    max_drawdown = drawdown.min()
    
    return {
        '策略': name,
        '总收益': f"{total_return:.2%}",
        '年化收益': f"{annual_return:.2%}",
        '年化波动': f"{annual_vol:.2%}",
        '夏普比': f"{sharpe:.2f}",
        '最大回撤': f"{max_drawdown:.2%}"
    }


# 计算各策略指标
strategies = [
    (momentum_returns, '动量策略'),
    (momentum_sl_returns, '动量+止损'),
    (bb_returns, '布林带策略'),
    (rsi_returns, 'RSI策略')
]

metrics_list = [calculate_metrics(ret, name) for ret, name in strategies]
metrics_df = pd.DataFrame(metrics_list)

print("策略绩效对比：")
print(metrics_df.to_string(index=False))

# 绘制累积收益对比
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# 累积收益
ax1 = axes[0]
for ret, name in strategies:
    cumulative = (1 + ret).cumprod()
    ax1.plot(cumulative.index, cumulative, label=name, linewidth=1.5)
ax1.set_xlabel('日期')
ax1.set_ylabel('累积收益')
ax1.set_title('策略累积收益对比')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 回撤对比
ax2 = axes[1]
for ret, name in strategies:
    cumulative = (1 + ret).cumprod()
    peak = cumulative.expanding().max()
    drawdown = (cumulative - peak) / peak
    ax2.fill_between(drawdown.index, drawdown, 0, alpha=0.3, label=name)
ax2.set_xlabel('日期')
ax2.set_ylabel('回撤')
ax2.set_title('策略回撤对比')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.2 适用市场环境

| 策略 | 适用环境 | 不适用环境 |
|------|----------|------------|
| **动量** | 趋势明显的市场 | 震荡市场、市场反转 |
| **布林带** | 震荡市场 | 强趋势市场 |
| **RSI** | 震荡市场 | 强趋势市场 |

**建议**：根据市场环境选择策略，或者组合使用多种策略。

---
## 7. 小结

### 核心收获

1. **动量策略**的金融逻辑是「强者恒强」，基于投资者反应不足

2. **均值回归策略**的金融逻辑是「价格回归」，基于套利机制

3. **动量崩溃**是动量策略的主要风险，需要止损机制

4. **布林带和 RSI**是常用的均值回归指标

5. **不同策略适用于不同市场环境**，需要灵活选择

### 关键公式

- **动量收益**：$MOM = \sum_{t-1}^{t-N} R_t$
- **布林带**：$Upper = MA + K \times \sigma$，$Lower = MA - K \times \sigma$
- **RSI**：$RSI = 100 - \frac{100}{1 + RS}$，$RS = \frac{平均上涨}{平均下跌}$

### 验收标准 Checklist

- [x] **能解释动量的金融逻辑**：投资者反应不足、正反馈循环
- [x] **理解动量崩溃风险**：市场反转、拥挤交易、流动性危机
- [x] **策略含仓位管理和止损**：实现了带止损的动量策略

---

**恭喜你完成了动量与均值回归策略的学习！** 🎉